# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import uuid
from datetime import datetime, timezone
from typing import TypedDict, List, Optional, Literal, Union

from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.llm import LLM
from lib.messages import SystemMessage, UserMessage, AIMessage
from lib.tooling import tool
from lib.state_machine import StateMachine, Step, EntryPoint, Termination
from lib.memory import ShortTermMemory
from lib.game_store import GameVectorStore

In [3]:
# find_dotenv() finds the .env file in this folder (or a parent folder);
# override=True makes the values in .env win over anything already set.
load_dotenv(find_dotenv(), override=True)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
print("Environment variables loaded")

Environment variables loaded


In [4]:
MODEL_NAME = "gpt-4o-mini"

# Below this evaluation confidence the agent falls back to the web search
CONFIDENCE_THRESHOLD = 0.7

N_GAME_RESULTS = 5    # documents retrieved from the local dataset
N_MEMORY_RESULTS = 3  # entries retrieved from the long-term memory

# Same manager as Part 1: same path, collection name and embedding function.
# create=False: fails if Part 1 has not been run.
store = GameVectorStore(path="chromadb", collection_name="udaplay", create=False)
chroma_client = store.client
embedding_fn = store.embedding_function
print(f"Loaded {store}")

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

Loaded GameVectorStore(collection='udaplay', documents=15)


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web

Two more tools back the long-term memory (see the Advanced section): `search_memory` and `store_memory`.

#### Retrieve Game Tool

In [5]:
@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds the most similar games in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...)
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    - Genre: Genre of the game
    - Publisher: Company that published the game
    - id: document id in the vector DB
    - distance: semantic distance from the query (lower is closer)
    """
    return store.search(query, n_results=N_GAME_RESULTS)

#### Evaluate Retrieval Tool

In [6]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are enough to answer the question")
    description: str = Field(description="Detailed explanation of the evaluation result")
    confidence: float = Field(description="From 0.0 to 1.0: how confident you are that the documents fully answer the question")


EVALUATION_PROMPT = (
    "Your task is to evaluate if the documents are enough to respond the query. "
    "Give a detailed explanation, so it's possible to take an action to accept it or not. "
    "The documents are useful only if they contain the specific facts the question asks for; "
    "documents about other games, or facts that may be outdated for questions about the "
    "present, are not enough."
)


@tool
def evaluate_retrieval(question: str, retrieved_docs: list[str]) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    - confidence: from 0.0 to 1.0, how confident the evaluator is that the documents answer the question
    """
    if not retrieved_docs:
        return EvaluationReport(
            useful=False, description="No documents were retrieved.", confidence=0.0
        ).model_dump()

    llm = LLM(model=MODEL_NAME, temperature=0.0)
    documents = "\n".join(f"- {doc}" for doc in retrieved_docs)
    response = llm.invoke(
        [
            SystemMessage(content=EVALUATION_PROMPT),
            UserMessage(content=f"Question: {question}\n\nDocuments:\n{documents}"),
        ],
        response_format=EvaluationReport,
    )
    return EvaluationReport.model_validate_json(response.content).model_dump()

#### Game Web Search Tool

In [7]:
@tool
def game_web_search(question: str) -> dict:
    """
    Web search: Finds up-to-date information on the web using the Tavily API.
    Use it when the internal knowledge is not enough to answer.
    args:
    - question: a question about game industry.

    The result includes:
    - answer: a short answer generated by Tavily from the pages found
    - results: list of web pages, each with title, url and content
    """
    response = tavily_client.search(
        query=question,
        search_depth="advanced",
        max_results=5,
        include_answer=True,
    )
    return {
        "answer": response.get("answer"),
        "results": [
            {"title": r.get("title"), "url": r.get("url"), "content": r.get("content")}
            for r in response.get("results", [])
        ],
    }

#### Long-term Memory Tools

What the agent learns from the web is saved in a second ChromaDB collection, `udaplay_memory`, in the same persistent database as the games. Unlike the conversation state, it survives new sessions, new agent instances and notebook restarts.

`RESET_LONG_TERM_MEMORY = True` empties it at the start of the notebook, so the demo below always shows the web fallback first and then the memory hit. Set it to `False` to keep what was learned in previous runs.

In [8]:
RESET_LONG_TERM_MEMORY = True

if RESET_LONG_TERM_MEMORY and "udaplay_memory" in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection("udaplay_memory")

memory_collection = chroma_client.get_or_create_collection(
    name="udaplay_memory",
    embedding_function=embedding_fn,
)
print(f"Long-term memory '{memory_collection.name}' has {memory_collection.count()} entries")

Long-term memory 'udaplay_memory' has 0 entries


In [9]:
@tool
def search_memory(query: str) -> list:
    """
    Semantic search over the long-term memory: facts the agent learned from previous web searches.
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - question: the question that was answered
    - answer: the answer that was learned
    - sources: URLs the answer came from
    - learned_at: when the answer was stored (UTC, ISO format)
    - distance: semantic distance from the query (lower is closer)
    """
    count = memory_collection.count()
    if count == 0:
        return []
    results = memory_collection.query(
        query_texts=[query],
        n_results=min(N_MEMORY_RESULTS, count),
        include=["metadatas", "distances"],
    )
    return [
        {
            "id": memory_id,
            "question": metadata["question"],
            "answer": metadata["answer"],
            "sources": json.loads(metadata["sources"]),
            "learned_at": metadata["learned_at"],
            "distance": round(distance, 4),
        }
        for memory_id, metadata, distance in zip(
            results["ids"][0], results["metadatas"][0], results["distances"][0]
        )
    ]


@tool
def store_memory(question: str, answer: str, source_urls: list[str]) -> str:
    """
    Stores an answer learned from the web in the long-term memory.
    args:
    - question: the question that was answered
    - answer: the answer to remember
    - source_urls: URLs the answer came from
    Returns the id of the new memory entry.
    """
    memory_id = str(uuid.uuid4())
    memory_collection.add(
        ids=[memory_id],
        documents=[f"Question: {question}\nAnswer: {answer}"],
        metadatas=[{
            "question": question,
            "answer": answer,
            "sources": json.dumps(source_urls),
            "learned_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        }],
    )
    return memory_id

### Agent

UdaPlay is a state machine (`lib/state_machine.py`) whose nodes are the tools, so every question follows the same path:

```
contextualize -> retrieve_game -> search_memory -> evaluate_retrieval
    -> game_web_search   (only if the documents are not useful, or confidence < CONFIDENCE_THRESHOLD)
    -> generate_answer
    -> store_memory      (only if the answer came from the web)
```

- **contextualize** rewrites follow-up questions ("who published it?") into standalone ones, using the previous turns of the session.
- **Conversation state**: each session keeps its runs in a `ShortTermMemory`; the previous questions and answers are fed to the next run.
- **generate_answer** returns a structured report: the answer with inline citations, a confidence level and the sources actually cited (with URLs for web sources).

In [10]:
class Source(BaseModel):
    id: str = Field(description="Evidence id, e.g. S1")
    origin: Literal["local_dataset", "long_term_memory", "web"]
    title: str
    urls: List[str] = Field(default_factory=list)


class UdaPlayReport(BaseModel):
    question: str
    standalone_question: str
    answer: str
    confidence: float
    used_web_search: bool
    sources: List[Source]
    retrieval_evaluation: EvaluationReport


class AnswerDraft(BaseModel):
    answer: str = Field(description="Answer in natural language, citing evidence inline like [S1]")
    confidence: float = Field(description="From 0.0 to 1.0: how well the evidence supports the answer")
    cited_sources: List[str] = Field(description="Ids of the evidence items used in the answer, e.g. ['S1', 'S3']")


class StandaloneQuestion(BaseModel):
    question: str


class UdaPlayState(TypedDict):
    session_id: str
    user_query: str
    history: List[dict]            # previous turns of the session: {"question", "answer"}
    standalone_query: str
    local_results: List[dict]
    memory_results: List[dict]
    evaluation: dict
    needs_web_search: bool
    web_results: Optional[dict]
    evidence: List[dict]           # numbered evidence passed to the answer step
    report: Optional[dict]
    memory_id: Optional[str]

In [11]:
ANSWER_INSTRUCTIONS = """You are UdaPlay, an AI research assistant for the video game industry.
Answer the user's question using ONLY the evidence provided.
- Cite the evidence inline with its id, like [S1]. List every id you used in cited_sources.
- Cite a source only for the facts it explicitly states. Never attach a citation to a fact
  that is not in that source's text (e.g. an older article cannot support a newer date).
- Prefer authoritative sources: official publisher, developer or platform pages, established
  news outlets and encyclopedias. Use videos, forums or shop listings only when nothing
  better supports the fact.
- For questions about the present, rely on the most recent information and mention when
  a fact may change (e.g. announced release dates).
- When several sources are relevant, combine them into one clear answer.
- If sources disagree, prefer web sources for recent facts and say there is a discrepancy.
- If the evidence does not answer the question, say so plainly and give a low confidence.
- Write a short, natural answer (a few sentences) in the user's language."""

CONTEXTUALIZE_INSTRUCTIONS = """Rewrite the user's latest question as a standalone question,
resolving references like "it", "that game" or "they" using the conversation.
If the question is already standalone, return it unchanged."""


class UdaPlayAgent:
    def __init__(self,
                 model_name: str = MODEL_NAME,
                 confidence_threshold: float = CONFIDENCE_THRESHOLD):
        self.model_name = model_name
        self.confidence_threshold = confidence_threshold
        self.memory = ShortTermMemory()
        self.workflow = self._create_state_machine()

    def _llm(self) -> LLM:
        return LLM(model=self.model_name, temperature=0.0)

    @staticmethod
    def _log(step: str, message: str):
        print(f"    [{step}] {message}")

    # --- Steps ---------------------------------------------------------------

    def _contextualize_step(self, state: UdaPlayState) -> UdaPlayState:
        query = state["user_query"]
        history = state.get("history", [])
        if not history:
            self._log("contextualize", f"New conversation, question used as is: {query!r}")
            return {"standalone_query": query}

        conversation = "\n".join(
            f"User: {turn['question']}\nUdaPlay: {turn['answer']}" for turn in history[-5:]
        )
        response = self._llm().invoke(
            [
                SystemMessage(content=CONTEXTUALIZE_INSTRUCTIONS),
                UserMessage(content=f"Conversation:\n{conversation}\n\nLatest question: {query}"),
            ],
            response_format=StandaloneQuestion,
        )
        standalone = StandaloneQuestion.model_validate_json(response.content).question
        self._log("contextualize", f"{len(history)} previous turn(s); standalone question: {standalone!r}")
        return {"standalone_query": standalone}

    def _retrieve_step(self, state: UdaPlayState) -> UdaPlayState:
        results = retrieve_game(state["standalone_query"])
        self._log("retrieve_game", f"{len(results)} documents from the local dataset:")
        for r in results:
            print(f"        {r['id']} {r['Name']} | {r['Platform']} | {r['YearOfRelease']} | distance={r['distance']}")
        return {"local_results": results}

    def _memory_step(self, state: UdaPlayState) -> UdaPlayState:
        results = search_memory(state["standalone_query"])
        self._log("search_memory", f"{len(results)} entries from the long-term memory")
        for r in results:
            print(f"        {r['question']!r} (learned {r['learned_at']}) | distance={r['distance']}")
        return {"memory_results": results}

    def _evaluate_step(self, state: UdaPlayState) -> UdaPlayState:
        docs = [
            f"[local_dataset] {json.dumps({k: v for k, v in r.items() if k != 'distance'}, ensure_ascii=False)}"
            for r in state["local_results"]
        ] + [
            f"[long_term_memory, learned {r['learned_at']}] Question: {r['question']} Answer: {r['answer']}"
            for r in state["memory_results"]
        ]
        evaluation = evaluate_retrieval(state["standalone_query"], docs)
        needs_web = (not evaluation["useful"]) or evaluation["confidence"] < self.confidence_threshold
        self._log(
            "evaluate_retrieval",
            f"useful={evaluation['useful']} confidence={evaluation['confidence']:.2f} "
            f"-> {'web search needed' if needs_web else 'internal knowledge is enough'}",
        )
        print(f"        reasoning: {evaluation['description']}")
        return {"evaluation": evaluation, "needs_web_search": needs_web}

    def _web_search_step(self, state: UdaPlayState) -> UdaPlayState:
        results = game_web_search(state["standalone_query"])
        self._log("game_web_search", f"{len(results['results'])} web pages found:")
        for r in results["results"]:
            print(f"        {r['title']} - {r['url']}")
        return {"web_results": results}

    def _build_evidence(self, state: UdaPlayState) -> List[dict]:
        evidence = []
        for r in state["local_results"]:
            evidence.append({
                "origin": "local_dataset",
                "title": f"{r['Name']} ({r['Platform']}, {r['YearOfRelease']}) - local dataset",
                "urls": [],
                "content": json.dumps({k: v for k, v in r.items() if k != "distance"}, ensure_ascii=False),
            })
        for r in state["memory_results"]:
            evidence.append({
                "origin": "long_term_memory",
                "title": f"Long-term memory (learned {r['learned_at']}): {r['question']}",
                "urls": r["sources"],
                "content": r["answer"],
            })
        web = state.get("web_results") or {}
        for r in web.get("results", []):
            evidence.append({
                "origin": "web",
                "title": r["title"],
                "urls": [r["url"]],
                "content": r["content"],
            })
        for i, item in enumerate(evidence, start=1):
            item["id"] = f"S{i}"
        return evidence

    def _generate_step(self, state: UdaPlayState) -> UdaPlayState:
        evidence = self._build_evidence(state)
        evidence_text = "\n\n".join(
            f"[{e['id']}] ({e['origin']}) {e['title']}"
            + (f"\nURL: {', '.join(e['urls'])}" if e["urls"] else "")
            + f"\n{e['content']}"
            for e in evidence
        )

        messages = [SystemMessage(content=ANSWER_INSTRUCTIONS)]
        for turn in state.get("history", [])[-5:]:
            messages.append(UserMessage(content=turn["question"]))
            messages.append(AIMessage(content=turn["answer"]))
        messages.append(UserMessage(content=(
            f"Question: {state['user_query']}\n"
            f"Standalone question: {state['standalone_query']}\n\n"
            f"Evidence:\n{evidence_text}"
        )))

        response = self._llm().invoke(messages, response_format=AnswerDraft)
        draft = AnswerDraft.model_validate_json(response.content)

        by_id = {e["id"]: e for e in evidence}
        sources = [
            Source(id=e["id"], origin=e["origin"], title=e["title"], urls=e["urls"])
            for e in (by_id[i] for i in dict.fromkeys(draft.cited_sources) if i in by_id)
        ]
        report = UdaPlayReport(
            question=state["user_query"],
            standalone_question=state["standalone_query"],
            answer=draft.answer,
            confidence=draft.confidence,
            used_web_search=state["needs_web_search"],
            sources=sources,
            retrieval_evaluation=EvaluationReport(**state["evaluation"]),
        )
        self._log("generate_answer", f"confidence={draft.confidence:.2f}, cited {[s.id for s in sources]}")
        return {"evidence": evidence, "report": report.model_dump()}

    def _store_memory_step(self, state: UdaPlayState) -> UdaPlayState:
        report = state["report"]
        urls = [url for s in report["sources"] if s["origin"] == "web" for url in s["urls"]]
        memory_id = store_memory(report["standalone_question"], report["answer"], urls)
        self._log("store_memory", f"Saved to long-term memory (id={memory_id}, {len(urls)} source URL(s))")
        return {"memory_id": memory_id}

    # --- Workflow ------------------------------------------------------------

    def _create_state_machine(self) -> StateMachine[UdaPlayState]:
        machine = StateMachine[UdaPlayState](UdaPlayState)

        entry = EntryPoint[UdaPlayState]()
        contextualize = Step[UdaPlayState]("contextualize", self._contextualize_step)
        retrieve = Step[UdaPlayState]("retrieve_game", self._retrieve_step)
        recall = Step[UdaPlayState]("search_memory", self._memory_step)
        evaluate = Step[UdaPlayState]("evaluate_retrieval", self._evaluate_step)
        web_search = Step[UdaPlayState]("game_web_search", self._web_search_step)
        generate = Step[UdaPlayState]("generate_answer", self._generate_step)
        remember = Step[UdaPlayState]("store_memory", self._store_memory_step)
        termination = Termination[UdaPlayState]()

        machine.add_steps([entry, contextualize, retrieve, recall, evaluate,
                           web_search, generate, remember, termination])

        machine.connect(entry, contextualize)
        machine.connect(contextualize, retrieve)
        machine.connect(retrieve, recall)
        machine.connect(recall, evaluate)

        def route_after_evaluation(state: UdaPlayState) -> Step[UdaPlayState]:
            return web_search if state["needs_web_search"] else generate

        def route_after_answer(state: UdaPlayState) -> Step[UdaPlayState]:
            return remember if state["needs_web_search"] else termination

        machine.connect(evaluate, [web_search, generate], route_after_evaluation)
        machine.connect(web_search, generate)
        machine.connect(generate, [remember, termination], route_after_answer)
        machine.connect(remember, termination)

        return machine

    # --- Public API ----------------------------------------------------------

    def get_history(self, session_id: Optional[str] = None) -> List[dict]:
        """Questions and answers of the previous runs in the session."""
        session_id = session_id or "default"
        if session_id not in self.memory.get_all_sessions():
            return []
        history = []
        for run in self.memory.get_all_objects(session_id):
            report = run.get_final_state()["report"]
            history.append({"question": report["question"], "answer": report["answer"]})
        return history

    def invoke(self, query: str, session_id: Optional[str] = None) -> UdaPlayReport:
        session_id = session_id or "default"
        history = self.get_history(session_id)
        self.memory.create_session(session_id)

        initial_state: UdaPlayState = {
            "session_id": session_id,
            "user_query": query,
            "history": history,
            "standalone_query": query,
            "local_results": [],
            "memory_results": [],
            "evaluation": {},
            "needs_web_search": False,
            "web_results": None,
            "evidence": [],
            "report": None,
            "memory_id": None,
        }

        print(f">>> [{session_id}] {query}")
        run = self.workflow.run(initial_state)
        self.memory.add(run, session_id)
        return UdaPlayReport(**run.get_final_state()["report"])

    def reset_session(self, session_id: Optional[str] = None):
        self.memory.reset(session_id)

#### Report

In [12]:
def print_report(report: UdaPlayReport):
    """Prints the final answer with its confidence and cited sources."""
    print("=" * 80)
    print(f"QUESTION: {report.question}")
    if report.standalone_question != report.question:
        print(f"Interpreted as: {report.standalone_question}")
    print(f"\nANSWER:\n{report.answer}\n")
    print(f"Confidence: {report.confidence:.0%} | "
          f"Retrieval evaluation: useful={report.retrieval_evaluation.useful}, "
          f"confidence={report.retrieval_evaluation.confidence:.0%} | "
          f"Web search used: {'yes' if report.used_web_search else 'no'}")
    print("\nSOURCES:")
    if not report.sources:
        print("  (none cited)")
    for s in report.sources:
        print(f"  [{s.id}] ({s.origin}) {s.title}")
        for url in s.urls:
            print(f"       {url}")
    print("=" * 80 + "\n")

### Run the agent

The three example queries: the first two can be answered by the local dataset, Mortal Kombat X is not in it and triggers the web fallback. For each query the output shows the steps (`[StateMachine]` lines and the log of each tool), then the final report.

In [13]:
agent = UdaPlayAgent()

questions = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for question in questions:
    print_report(agent.invoke(question, session_id="demo"))

>>> [demo] When was Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
    [contextualize] New conversation, question used as is: 'When was Pokémon Gold and Silver released?'
[StateMachine] Executing step: contextualize


    [retrieve_game] 5 documents from the local dataset:
        006 Pokémon Gold and Silver | Game Boy Color | 1999 | distance=0.1184
        007 Pokémon Ruby and Sapphire | Game Boy Advance | 2002 | distance=0.1508
        009 Super Mario 64 | Nintendo 64 | 1996 | distance=0.2295
        008 Super Mario World | Super Nintendo Entertainment System (SNES) | 1990 | distance=0.2398
        011 Wii Sports | Wii | 2006 | distance=0.2477
[StateMachine] Executing step: retrieve_game
    [search_memory] 0 entries from the long-term memory
[StateMachine] Executing step: search_memory


    [evaluate_retrieval] useful=True confidence=1.00 -> internal knowledge is enough
        reasoning: The documents contain specific information about the release date of Pokémon Gold and Silver, stating that it was released in 1999. This directly answers the query regarding when Pokémon Gold and Silver was released. The other documents do not pertain to the question and are about different games, but the relevant document provides the necessary information.
[StateMachine] Executing step: evaluate_retrieval


    [generate_answer] confidence=1.00, cited ['S1']
[StateMachine] Executing step: generate_answer
[StateMachine] Terminating: __termination__
QUESTION: When was Pokémon Gold and Silver released?

ANSWER:
Pokémon Gold and Silver were released in 1999 for the Game Boy Color [S1].

Confidence: 100% | Retrieval evaluation: useful=True, confidence=100% | Web search used: no

SOURCES:
  [S1] (local_dataset) Pokémon Gold and Silver (Game Boy Color, 1999) - local dataset

>>> [demo] Which one was the first 3D platformer Mario game?
[StateMachine] Starting: __entry__


    [contextualize] 1 previous turn(s); standalone question: 'Which game was the first 3D platformer in the Mario series?'
[StateMachine] Executing step: contextualize


...

    [retrieve_game] 5 documents from the local dataset:
        005 Marvel's Spider-Man 2 | PlayStation 5 | 2023 | distance=0.2143
        004 Marvel's Spider-Man | PlayStation 4 | 2018 | distance=0.228
        015 Halo Infinite | Xbox Series X|S | 2021 | distance=0.2284
        003 Gran Turismo 5 | PlayStation 3 | 2010 | distance=0.2309
        013 Kinect Adventures! | Xbox 360 | 2010 | distance=0.2519
[StateMachine] Executing step: retrieve_game
    [search_memory] 0 entries from the long-term memory
[StateMachine] Executing step: search_memory


    [evaluate_retrieval] useful=False confidence=0.00 -> web search needed
        reasoning: The documents provided do not contain any information regarding the release of Mortal Kombat X for PlayStation 5. Instead, they focus on other games such as Marvel's Spider-Man 2, Marvel's Spider-Man, Halo Infinite, Gran Turismo 5, and Kinect Adventures!, none of which are related to Mortal Kombat X. Additionally, there is no mention of Mortal Kombat X or its platforms in any of the documents. Therefore, they are not sufficient to answer the question about Mortal Kombat X's release on PlayStation 5.
[StateMachine] Executing step: evaluate_retrieval


    [game_web_search] 5 web pages found:
        Mortal Kombat X - PS5 Gameplay - https://www.youtube.com/watch?v=tqsw711ZuAk&xstg=CAMSBhUDze3xHw%3D%3D
        Mortal Kombat X - PS5 Gameplay - https://www.youtube.com/watch?v=tqsw711ZuAk
        Mortal Kombat PS Games | MK 1, MK X, PS5, PS4 - https://www.target.com/s/mortal+kombat+ps
        PS5 PlayStation Plus Collection To Include Mortal Kombat X - Mortal Kombat Online - https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V
        Mortal Kombat X - https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved
[StateMachine] Executing step: game_web_search


    [generate_answer] confidence=0.90, cited ['S9', 'S10']
[StateMachine] Executing step: generate_answer


    [store_memory] Saved to long-term memory (id=05e67d35-d953-47f6-a204-dc678c8f3004, 2 source URL(s))
[StateMachine] Executing step: store_memory
[StateMachine] Terminating: __termination__
QUESTION: Was Mortal Kombat X released for PlayStation 5?

ANSWER:
Mortal Kombat X was not originally released for PlayStation 5, but it is playable on the console through backward compatibility. Additionally, it is included in the PlayStation Plus Collection, allowing subscribers to access it on PS5 [S9][S10].

Confidence: 90% | Retrieval evaluation: useful=False, confidence=0% | Web search used: yes

SOURCES:
  [S9] (web) PS5 PlayStation Plus Collection To Include Mortal Kombat X - Mortal Kombat Online
       https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V
  [S10] (web) Mortal Kombat X
       https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved



#### Conversation state

Within a session the agent remembers the previous questions: the follow-ups below only make sense with the context of the earlier turns. The last one needs current information, so it goes to the web.

In [14]:
conversation = [
    "Who published Grand Theft Auto: San Andreas?",
    "On which platform was it released, and in what year?",
    "What is that publisher working on right now?",
]

for question in conversation:
    print_report(agent.invoke(question, session_id="conversation"))

>>> [conversation] Who published Grand Theft Auto: San Andreas?
[StateMachine] Starting: __entry__
    [contextualize] New conversation, question used as is: 'Who published Grand Theft Auto: San Andreas?'
[StateMachine] Executing step: contextualize


    [retrieve_game] 5 documents from the local dataset:
        002 Grand Theft Auto: San Andreas | PlayStation 2 | 2004 | distance=0.1233
        001 Gran Turismo | PlayStation 1 | 1997 | distance=0.2101
        003 Gran Turismo 5 | PlayStation 3 | 2010 | distance=0.2133
        009 Super Mario 64 | Nintendo 64 | 1996 | distance=0.232
        004 Marvel's Spider-Man | PlayStation 4 | 2018 | distance=0.2321
[StateMachine] Executing step: retrieve_game


    [search_memory] 1 entries from the long-term memory
        'Was Mortal Kombat X released for PlayStation 5?' (learned 2026-10-20T07:01:02+00:00) | distance=0.2782
[StateMachine] Executing step: search_memory


    [evaluate_retrieval] useful=True confidence=1.00 -> internal knowledge is enough
        reasoning: The documents contain specific information about the game 'Grand Theft Auto: San Andreas', including its publisher, which is Rockstar Games. This directly answers the query regarding who published the game. The other documents pertain to different games and do not provide relevant information for the question, but the first document is sufficient on its own to confirm the publisher of 'Grand Theft Auto: San Andreas'.
[StateMachine] Executing step: evaluate_retrieval


    [generate_answer] confidence=1.00, cited ['S1']
[StateMachine] Executing step: generate_answer
[StateMachine] Terminating: __termination__
QUESTION: Who published Grand Theft Auto: San Andreas?

ANSWER:
Grand Theft Auto: San Andreas was published by Rockstar Games [S1].

Confidence: 100% | Retrieval evaluation: useful=True, confidence=100% | Web search used: no

SOURCES:
  [S1] (local_dataset) Grand Theft Auto: San Andreas (PlayStation 2, 2004) - local dataset

>>> [conversation] On which platform was it released, and in what year?
[StateMachine] Starting: __entry__


...

    [search_memory] 1 entries from the long-term memory
        'Was Mortal Kombat X released for PlayStation 5?' (learned 2026-10-20T07:01:02+00:00) | distance=0.2662
[StateMachine] Executing step: search_memory


    [evaluate_retrieval] useful=False confidence=0.00 -> web search needed
        reasoning: The documents provided do not contain any information about what Rockstar Games is currently working on. They only include details about past games published by Rockstar Games, specifically 'Grand Theft Auto: San Andreas', and other unrelated games published by different companies. Since the question specifically asks for current projects or developments from Rockstar Games, and none of the documents address this, they are not useful for answering the query.
[StateMachine] Executing step: evaluate_retrieval


    [game_web_search] 5 web pages found:
        Grand Theft Auto: San Andreas - Wikipedia - https://en.wikipedia.org/wiki/Grand_Theft_Auto:_San_Andreas
        Rockstar Games Is Releasing A New Game In 2026... (Not GTA 6) - https://www.youtube.com/watch?v=EIz4UFrfUfg
        Rockstar Games' Future - Rockstar Games - GTAForums - https://gtaforums.com/topic/1000314-rockstar-games-future
        Grand Theft Auto 6: Rockstar announces it is working on the game - https://www.bbc.com/news/entertainment-arts-60266275
        Rockstar's next game won't take eight years to make: "The engine, the tools and the team we've built during GTA 6 will allow us to do things more quickly" | GamesIndustry.biz - https://www.gamesindustry.biz/rockstars-next-game-wont-take-eight-years-to-make-the-engine-the-tools-and-the-team-weve-built-during-gta-6-will-allow-us-to-do-things-more-quickly
[StateMachine] Executing step: game_web_search


    [generate_answer] confidence=0.90, cited ['S10', 'S11']
[StateMachine] Executing step: generate_answer


    [store_memory] Saved to long-term memory (id=1ea402ef-84e1-4e80-8314-3da98d99fb88, 2 source URL(s))
[StateMachine] Executing step: store_memory
[StateMachine] Terminating: __termination__
QUESTION: What is that publisher working on right now?
Interpreted as: What is Rockstar Games, the publisher of Grand Theft Auto: San Andreas, working on right now?

ANSWER:
Rockstar Games is currently working on Grand Theft Auto VI, which is in active development and is expected to be released on November 19, 2026 [S10][S11].

Confidence: 90% | Retrieval evaluation: useful=False, confidence=0% | Web search used: yes

SOURCES:
  [S10] (web) Grand Theft Auto 6: Rockstar announces it is working on the game
       https://www.bbc.com/news/entertainment-arts-60266275
  [S11] (web) Rockstar's next game won't take eight years to make: "The engine, the tools and the team we've built during GTA 6 will allow us to do things more quickly" | GamesIndustry.biz
       https://www.gamesindustry.biz/rockstars-ne

In [15]:
for session_id in ["demo", "conversation"]:
    print(f"Session '{session_id}':")
    for i, turn in enumerate(agent.get_history(session_id), start=1):
        print(f"  {i}. {turn['question']}")

Session 'demo':
  1. When was Pokémon Gold and Silver released?
  2. Which one was the first 3D platformer Mario game?
  3. Was Mortal Kombat X released for PlayStation 5?
Session 'conversation':
  1. Who published Grand Theft Auto: San Andreas?
  2. On which platform was it released, and in what year?
  3. What is that publisher working on right now?


### (Optional) Advanced

#### Long-term memory

A **new agent instance** has an empty conversation state, but it shares the persistent `udaplay_memory` collection. Asking again about Mortal Kombat X, the evaluation accepts what was learned from the web earlier, so no new web search is made and the answer cites the memory (with the original URLs).

In [16]:
print(f"Long-term memory entries: {memory_collection.count()}")
for metadata in memory_collection.get(include=["metadatas"])["metadatas"]:
    print(f"- {metadata['question']} (learned {metadata['learned_at']})")
    print(f"  sources: {', '.join(json.loads(metadata['sources']))}")

Long-term memory entries: 2
- Was Mortal Kombat X released for PlayStation 5? (learned 2026-10-20T07:01:02+00:00)
  sources: https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V, https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved
- What is Rockstar Games, the publisher of Grand Theft Auto: San Andreas, working on right now? (learned 2026-10-20T07:01:25+00:00)
  sources: https://www.bbc.com/news/entertainment-arts-60266275, https://www.gamesindustry.biz/rockstars-next-game-wont-take-eight-years-to-make-the-engine-the-tools-and-the-team-weve-built-during-gta-6-will-allow-us-to-do-things-more-quickly


In [17]:
new_agent = UdaPlayAgent()
memory_report = new_agent.invoke("Was Mortal Kombat X released for PlayStation 5?", session_id="new_session")
print_report(memory_report)

>>> [new_session] Was Mortal Kombat X released for PlayStation 5?
[StateMachine] Starting: __entry__
    [contextualize] New conversation, question used as is: 'Was Mortal Kombat X released for PlayStation 5?'
[StateMachine] Executing step: contextualize


    [retrieve_game] 5 documents from the local dataset:
        005 Marvel's Spider-Man 2 | PlayStation 5 | 2023 | distance=0.2143
        004 Marvel's Spider-Man | PlayStation 4 | 2018 | distance=0.228
        015 Halo Infinite | Xbox Series X|S | 2021 | distance=0.2284
        003 Gran Turismo 5 | PlayStation 3 | 2010 | distance=0.2309
        013 Kinect Adventures! | Xbox 360 | 2010 | distance=0.2519
[StateMachine] Executing step: retrieve_game


    [search_memory] 2 entries from the long-term memory
        'Was Mortal Kombat X released for PlayStation 5?' (learned 2026-10-20T07:01:02+00:00) | distance=0.0694
        'What is Rockstar Games, the publisher of Grand Theft Auto: San Andreas, working on right now?' (learned 2026-10-20T07:01:25+00:00) | distance=0.248
[StateMachine] Executing step: search_memory


    [evaluate_retrieval] useful=True confidence=0.90 -> internal knowledge is enough
        reasoning: The documents do not contain any information about Mortal Kombat X or its availability on PlayStation 5. However, the long-term memory entries provide a clear answer to the question, stating that Mortal Kombat X was not originally released for PlayStation 5 but is playable on the console through backward compatibility and is included in the PlayStation Plus Collection. This information directly addresses the query about the game's release status on the PS5.
[StateMachine] Executing step: evaluate_retrieval


    [generate_answer] confidence=0.90, cited ['S6']
[StateMachine] Executing step: generate_answer
[StateMachine] Terminating: __termination__
QUESTION: Was Mortal Kombat X released for PlayStation 5?

ANSWER:
Mortal Kombat X was not originally released for PlayStation 5, but it is playable on the console through backward compatibility. Additionally, it is included in the PlayStation Plus Collection, allowing subscribers to access it on PS5 [S6].

Confidence: 90% | Retrieval evaluation: useful=True, confidence=90% | Web search used: no

SOURCES:
  [S6] (long_term_memory) Long-term memory (learned 2026-10-20T07:01:02+00:00): Was Mortal Kombat X released for PlayStation 5?
       https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V
       https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved



#### Structured output

Every answer is also available as structured JSON, ready for other systems to consume.

In [18]:
print(memory_report.model_dump_json(indent=2))

{
  "question": "Was Mortal Kombat X released for PlayStation 5?",
  "standalone_question": "Was Mortal Kombat X released for PlayStation 5?",
  "answer": "Mortal Kombat X was not originally released for PlayStation 5, but it is playable on the console through backward compatibility. Additionally, it is included in the PlayStation Plus Collection, allowing subscribers to access it on PS5 [S6].",
  "confidence": 0.9,
  "used_web_search": false,
  "sources": [
    {
      "id": "S6",
      "origin": "long_term_memory",
      "title": "Long-term memory (learned 2026-10-20T07:01:02+00:00): Was Mortal Kombat X released for PlayStation 5?",
      "urls": [
        "https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V",
        "https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved"
      ]
    }
  ],
  "retrieval_evaluation": {
    "useful": true,
    "description": "The documents do not contain any information about Mor